# Notebook 02 - Data Cleaning Pipeline
cleaning the raw dataset step by step


In [ ]:
import pandas as pd
import numpy as np
import json


loading raw data


In [ ]:
df = pd.read_csv('../data/raw/nyc_parking_tickets_sample.csv', dtype=str)
df.columns = df.columns.str.replace(' ', '_')
print("starting with", len(df), "rows and", len(df.columns), "columns")


step 1 - parsing dates and creating time based columns


In [ ]:
df['Issue_Date'] = pd.to_datetime(df['Issue_Date'], format='%m/%d/%Y', errors='coerce')
df['Month'] = df['Issue_Date'].dt.month
df['Month_Name'] = df['Issue_Date'].dt.month_name()
df['Quarter'] = df['Issue_Date'].dt.quarter
df['Day_of_Week'] = df['Issue_Date'].dt.day_name()
df['Day_Num'] = df['Issue_Date'].dt.dayofweek
df['Week_Num'] = df['Issue_Date'].dt.isocalendar().week
df['Is_Weekend'] = df['Day_Num'].isin([5, 6]).astype(int)
print("dates parsed, missing dates:", df['Issue_Date'].isnull().sum())


step 2 - fixing fine amounts. negatives and zeros dont make sense for parking fines


In [ ]:
df['Fine_Amount'] = pd.to_numeric(df['Fine_Amount'], errors='coerce')
df.loc[df['Fine_Amount'] <= 0, 'Fine_Amount'] = np.nan
median_by_violation = df.groupby('Violation_Description')['Fine_Amount'].transform('median')
df['Fine_Amount'] = df['Fine_Amount'].fillna(median_by_violation)
df['Fine_Category'] = pd.cut(df['Fine_Amount'], bins=[0, 50, 100, float('inf')], labels=['Low', 'Medium', 'High'])
print("fines fixed, remaining nulls:", df['Fine_Amount'].isnull().sum())


step 3 - vehicle year validation. removing impossible years like 9999


In [ ]:
df['Vehicle_Year'] = pd.to_numeric(df['Vehicle_Year'], errors='coerce')
df.loc[(df['Vehicle_Year'] == 9999) | (df['Vehicle_Year'] > 2014) | (df['Vehicle_Year'] < 1980), 'Vehicle_Year'] = np.nan
df['Vehicle_Age'] = 2014 - df['Vehicle_Year']
df['Vehicle_Age_Group'] = pd.cut(df['Vehicle_Age'], bins=[-1, 5, 10, 15, float('inf')], labels=['0-5', '6-10', '11-15', '16+'])


step 4 - standardizing vehicle colors


In [ ]:
color_map = {'GY': 'GRAY', 'BK': 'BLACK', 'WH': 'WHITE'}
df['Vehicle_Color'] = df['Vehicle_Color'].str.upper().replace(color_map)


step 5 - fixing borough names


In [ ]:
df['Violation_County'] = df['Violation_County'].str.title()


step 6 - consolidating vehicle makes. keeping top 15 and grouping rest as OTHER


In [ ]:
top_makes = df['Vehicle_Make'].value_counts().nlargest(15).index
df['Vehicle_Make'] = np.where(df['Vehicle_Make'].isin(top_makes), df['Vehicle_Make'], 'OTHER')


step 7 - grouping body types into broader categories


In [ ]:
passenger = ['SUBN', '4DSD', '2DSD']
commercial = ['VAN', 'DELV', 'PICK']
df['Vehicle_Type_Group'] = 'OTHER'
df.loc[df['Vehicle_Body_Type'].isin(passenger), 'Vehicle_Type_Group'] = 'Passenger'
df.loc[df['Vehicle_Body_Type'].isin(commercial), 'Vehicle_Type_Group'] = 'Commercial'


step 8 - validating registration states and flagging out of state plates


In [ ]:
valid_states = ['NY', 'NJ', 'PA', 'CT', 'FL', 'MA', 'TX', 'VA', 'MD', 'NC']
df.loc[~df['Registration_State'].isin(valid_states), 'Registration_State'] = 'UNKNOWN'
df['Is_OutOfState'] = (df['Registration_State'] != 'NY').astype(int)
df['State_Group'] = np.where(df['Registration_State'] == 'NY', 'NY', np.where(df['Registration_State'].isin(['NJ', 'CT']), 'Tri-State', 'Other'))


step 9 - plate category


In [ ]:
df['Plate_Category'] = np.where(df['Plate_Type'] == 'PAS', 'Passenger', 'Other')


step 10 - cleaning street names and extracting street type


In [ ]:
df['Street_Name'] = df['Street_Name'].str.upper().str.strip()
df['Street_Type'] = df['Street_Name'].str.split().str[-1]


adding street level aggregations


In [ ]:
street_stats = df.groupby('Street_Name').agg(
    Street_Total_Revenue=('Fine_Amount', 'sum'),
    Street_Ticket_Count=('Summons_Number', 'count')
).reset_index()
df = df.merge(street_stats, on='Street_Name', how='left')


step 11 - identifying repeat offenders based on plate frequency


In [ ]:
plate_counts = df['Plate_ID'].value_counts()
df['Ticket_Count_Per_Plate'] = df['Plate_ID'].map(plate_counts).fillna(0)
df['Is_Repeat_Offender'] = (df['Ticket_Count_Per_Plate'] >= 2).astype(int)
df['Offender_Tier'] = pd.cut(df['Ticket_Count_Per_Plate'], bins=[0, 2, 5, float('inf')], labels=['1-2', '3-5', '6+'])


adding violation severity and avoidability flags


In [ ]:
desc_lower = df['Violation_Description'].str.lower()
safety_keywords = ['fire hydrant', 'double parking', 'crosswalk']
df['Violation_Severity'] = np.where(desc_lower.isin(safety_keywords), 'Safety-Critical', 'Standard')
avoidable_keywords = ['no parking', 'street cleaning', 'expired', 'no standing']
df['Is_Avoidable'] = desc_lower.str.contains('|'.join(avoidable_keywords), na=False).astype(int)


data completeness scoring


In [ ]:
df['Data_Completeness_Score'] = df.notnull().sum(axis=1)
df['Is_Complete_Record'] = (df['Data_Completeness_Score'] > 20).astype(int)


step 12 - removing duplicates


In [ ]:
df = df.drop_duplicates(subset=['Summons_Number'])
print("after dedup:", len(df), "rows")


saving transformation log


In [ ]:
log = {
    "step_01": "parsed dates, created month/quarter/weekend columns",
    "step_02": "fixed negative fines, imputed with median per violation",
    "step_03": "removed impossible vehicle years",
    "step_04": "standardized color abbreviations",
    "step_05": "title cased borough names",
    "step_06": "consolidated vehicle makes to top 15",
    "step_07": "grouped body types",
    "step_08": "validated states, added out of state flag",
    "step_09": "cleaned streets, extracted suffix",
    "step_10": "flagged repeat offenders with tiers",
    "step_11": "added severity and avoidability",
    "step_12": "removed duplicate summons numbers"
}
with open('../docs/etl_transformation_log.json', 'w') as f:
    json.dump(log, f, indent=2)


saving cleaned dataset


In [ ]:
df.to_csv('../data/processed/nyc_parking_clean.csv', index=False)
print("final shape:", df.shape[0], "rows,", df.shape[1], "columns")
print("done")
